### Libraries

In [46]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd



#### Load, Clean and Merge Bike Sensors Data with Sites and Directions

In [47]:
#Load Sites
site_columns = ["sensor_id", "site_nr", "longitude", "latitude", "name",
                "domain", "road_number", "district", "municipality", "interval", "installation_date"]
sites = pd.read_csv("../data/raw/sites.csv", header=None, names=site_columns)
print(f"Sites: {len(sites):,} rows")

#Load directions
direction_columns = ["sensor_id", "direction", "direction_name"]
directions = pd.read_csv("../data/raw/richtingen.csv", header=None, names=direction_columns)
print(f"Directions: {len(directions):,} rows")


Sites: 151 rows
Directions: 305 rows


In [48]:
#Load Bike Sensors
folder = Path("../data/raw/sensors")
files = list(folder.glob("data-*.csv"))

column_names = ["sensor_id", "direction", "vehicle_type", "start_time", "end_time", "count"]

dfs = []
for file in files:
    df = pd.read_csv(file, header=None, names=column_names)
    
    # Filter cyclists only
    df = df[df["vehicle_type"] == "FIETSERS"]

    #Drop problematic sensors : 
    # 123: 73% of null counts
    # 142: test sensor, located in the same place as 52, with more than 10% null counts
    # 144: 100% of null counts
    df = df[~df["sensor_id"].isin([ 123, 142, 144,])]

    #Drop nulls
    df = df.dropna(subset=["count"])

    #Convert to datatime 
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["hour"] = df["start_time"].dt.floor("h")

    #Group by hor
    df = df.groupby(["sensor_id", "direction", "hour"], as_index=False)["count"].sum()


    #Merge with sites
    df = df.merge(sites, on="sensor_id", how="left")

    #Merge with directions
    df = df.merge(directions, on=["sensor_id", "direction"], how="left")
    
    dfs.append(df)

#Concatenate data
bike_data = pd.concat(dfs, ignore_index=True)


print(f"Total rows: {len(bike_data):,}")
print(f"Total columns: {bike_data.shape[1]}")





Total rows: 6,992,485
Total columns: 15


#### Load and Clean Accidents Data

In [49]:
accidents = pd.read_excel("../data/raw/OPENDATA_MAP_2017-2024.xlsx")

# Filter Flanders + bikes
accidents = accidents[accidents["TX_RGN_COLLISION_NL"] == "Vlaams Gewest"]
accidents = accidents[
    (accidents["TX_ROAD_USR_TYPE1_NL"] == "Fiets") |
    (accidents["TX_ROAD_USR_TYPE2_NL"] == "Fiets")
]
print(f"After Flanders + bike filter: {len(accidents):,} rows")

#Filter year>= 2020
accidents = accidents[accidents["DT_YEAR_COLLISION"] >= 2020]
print(f"After year filter (2020-2024): {len(accidents):,} rows")

# Drop unknown hour
accidents = accidents[accidents["DT_TIME"] != 99]
print(f"After dropping unknown hour: {len(accidents):,} rows")

# Drop missing coordinates
accidents = accidents.dropna(subset=["MS_X_COORD", "MS_Y_COORD"])
print(f"After dropping missing coordinates: {len(accidents):,} rows")

# Keep only relevant columns
columns_to_keep = [
    "DT_YEAR_COLLISION", "DT_MONTH_COLLISION", "DT_TIME",
    "MS_X_COORD", "MS_Y_COORD", "TX_MUNTY_COLLISION_NL",
    "TX_CROSSWAY_NL", "CD_ROAD_TYPE_NL", "TX_BUILD_UP_AREA_NL",
    "TX_WEATHER_NL", "TX_ROAD_CONDITION_NL", "TX_LIGHT_CONDITION_NL",
    "TX_CLASS_ACCIDENTS_NL", "TX_COLLISION_TYPE_NL",
    "TX_ROAD_USR_TYPE1_NL", "TX_ROAD_USR_TYPE2_NL", "TX_OBSTACLES_NL"
]
accidents = accidents[columns_to_keep]
print(f"Columns kept: {len(accidents.columns)}")

After Flanders + bike filter: 59,683 rows
After year filter (2020-2024): 38,275 rows
After dropping unknown hour: 38,275 rows
After dropping missing coordinates: 35,195 rows
Columns kept: 17


In [50]:
bike_data.head()

,sensor_id,direction,hour,count,site_nr,longitude,latitude,name,domain,road_number,district,municipality,interval,installation_date,direction_name
0,1,IN,2020-01-01 00:00:00,0.0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,Machelen Cyclists rich. Brucargo
1,1,IN,2020-01-01 01:00:00,0.0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,Machelen Cyclists rich. Brucargo
2,1,IN,2020-01-01 02:00:00,0.0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,Machelen Cyclists rich. Brucargo
3,1,IN,2020-01-01 03:00:00,51.0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,Machelen Cyclists rich. Brucargo
4,1,IN,2020-01-01 04:00:00,2.0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,Machelen Cyclists rich. Brucargo


In [51]:
accidents_with_sensor.head()

,MS_X_COORD,MS_Y_COORD,accident_count,geometry,index_right,sensor_id,longitude,latitude,name,municipality,distance_to_sensor
0,23392.100000,195557.000000,1,POINT (23392.1 195557),62,63,2.74289,51.13529,Nieuwpoort teller 3,Nieuwpoort,15374.803104
1,24568.510730,198144.960035,1,POINT (24568.511 198144.96),62,63,2.74289,51.13529,Nieuwpoort teller 3,Nieuwpoort,13052.102075
2,24865.938000,200075.056000,1,POINT (24865.938 200075.056),62,63,2.74289,51.13529,Nieuwpoort teller 3,Nieuwpoort,12019.021720
3,25203.339063,199881.303212,1,POINT (25203.339 199881.303),62,63,2.74289,51.13529,Nieuwpoort teller 3,Nieuwpoort,11768.335684
4,25217.520000,199971.126000,1,POINT (25217.52 199971.126),62,63,2.74289,51.13529,Nieuwpoort teller 3,Nieuwpoort,11723.531323


#### Spatial Join (Accidents + Sensors)

In [52]:
# Get unique sensor locations
unique_sensors = bike_data.drop_duplicates(subset=["sensor_id"])[
    ["sensor_id", "longitude", "latitude", "name", "municipality"]
].reset_index(drop=True)
print(f"Unique sensors: {len(unique_sensors)}")

# Agregate accidents by location
accidents_aggregated = accidents.groupby(
    ["MS_X_COORD", "MS_Y_COORD"]
).size().reset_index(name="accident_count")
print(f"Unique accident locations: {len(accidents_aggregated):,}")
print(f"Total accidents: {accidents_aggregated['accident_count'].sum():,}")
print(f"Accidents per location — min: {accidents_aggregated['accident_count'].min()}, "
      f"max: {accidents_aggregated['accident_count'].max()}, "
      f"mean: {accidents_aggregated['accident_count'].mean():.2f}")


# Convert accidents to GeoDataFrame in the Belgian coordinate system EPSG:31370
accidents_gdf = gpd.GeoDataFrame(
    accidents_aggregated,
    geometry=gpd.points_from_xy(accidents_aggregated["MS_X_COORD"], accidents_aggregated["MS_Y_COORD"]),
    crs="EPSG:31370"
).reset_index(drop=True)

print(f"Accidents GeoDataFrame: {len(accidents_gdf):,} rows | CRS: {accidents_gdf.crs}")

# Convert sensors to GeoDataFrame and reproject to EPSG:31370
sensors_gdf = gpd.GeoDataFrame(
    unique_sensors,
    geometry=gpd.points_from_xy(unique_sensors["longitude"], unique_sensors["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:31370").reset_index(drop=True)

print(f"Sensors GeoDataFrame: {len(sensors_gdf):,} rows | CRS: {sensors_gdf.crs}")
print(f"Sample coordinates after reprojection:")
print(f"{sensors_gdf.geometry.head(3).tolist()}")

# Spatial join - find nearest sensor for each accident
accidents_with_sensor = gpd.sjoin_nearest(
    accidents_gdf,
    sensors_gdf,
    how="left",
    distance_col="distance_to_sensor"
).reset_index(drop=True)
print(f"After sjoin_nearest: {len(accidents_with_sensor):,} rows")

# Only sensors with accidents whithin 500 m
accidents_near_sensor = accidents_with_sensor[
    accidents_with_sensor["distance_to_sensor"] <= 500
].reset_index(drop=True)
print(f"Accidents within 500m: {len(accidents_near_sensor):,} rows")
print(f"Sensors with at least 1 accident location: {accidents_near_sensor['sensor_id'].nunique()}")


Unique sensors: 139
Unique accident locations: 32,634
Total accidents: 35,195
Accidents per location — min: 1, max: 38, mean: 1.08
Accidents GeoDataFrame: 32,634 rows | CRS: EPSG:31370
Sensors GeoDataFrame: 139 rows | CRS: EPSG:31370
Sample coordinates after reprojection:
[<POINT (156143.904 178432.685)>, <POINT (157182.957 218365.646)>, <POINT (157219.957 218355.684)>]
After sjoin_nearest: 33,269 rows
Accidents within 500m: 748 rows
Sensors with at least 1 accident location: 111


In [53]:
# Analize duplicates
total = len(accidents_near_sensor)
unique_locations = accidents_near_sensor[["MS_X_COORD", "MS_Y_COORD"]].drop_duplicates()
duplicated = accidents_near_sensor[
    accidents_near_sensor.duplicated(subset=["MS_X_COORD", "MS_Y_COORD"], keep=False)
]

print(f"Total accident locations within 500m: {total}")
print(f"Unique locations: {len(unique_locations)}")
print(f"Duplicated locations: {total - len(unique_locations)}")
print(f"Sensors involved in duplicates: {sorted(duplicated['sensor_id'].unique())}")

print(f"\nDuplicated locations detail:")
print(duplicated[["MS_X_COORD", "MS_Y_COORD", "sensor_id", "distance_to_sensor"]]
      .sort_values(["MS_X_COORD", "MS_Y_COORD"]))

Total accident locations within 500m: 748
Unique locations: 735
Duplicated locations: 13
Sensors involved in duplicates: [np.int64(116), np.int64(117)]

Duplicated locations detail:
        MS_X_COORD     MS_Y_COORD  sensor_id  distance_to_sensor
268  110170.112100  187287.939200        117          498.588947
269  110170.112100  187287.939200        116          498.588947
270  110228.078300  188202.143700        116          471.877206
271  110228.078300  188202.143700        117          471.877206
272  110286.859689  187502.565604        117          257.034246
273  110286.859689  187502.565604        116          257.034246
274  110300.276983  188140.326722        116          395.801861
275  110300.276983  188140.326722        117          395.801861
276  110324.184300  188119.863100        116          372.542412
277  110324.184300  188119.863100        117          372.542412
278  110339.455602  188106.346465        116          357.897486
279  110339.455602  188106.346465     

In [54]:
#Explore duplicated sensors
print(sites[sites["sensor_id"].isin([116, 117])][
    ["sensor_id", "name", "municipality", "longitude", "latitude"]
])

     sensor_id            name municipality  longitude   latitude
115        116  Melle teller 2        Melle    3.80406  50.998611
116        117  Melle teller 1        Melle    3.80406  50.998611


We already new this sensors have the same location from the EDA, as they mesure different things, we will leave them as they are, this means, that ccident locations within 500m will be attributed to both sensors.

**CHECK IF THIS IS THE RIGHT DECISION

In [55]:
accidents_per_sensor = accidents_near_sensor.groupby(
    "sensor_id"
)["accident_count"].sum().reset_index()

print(f"Sensors with accidents: {len(accidents_per_sensor)}")
print(f"Total accidents matched: {accidents_per_sensor['accident_count'].sum():,}")
print(accidents_per_sensor.sort_values("accident_count", ascending=False).head(10))

Sensors with accidents: 111
Total accidents matched: 805
     sensor_id  accident_count
54          69              77
110        143              65
109        140              41
20          25              24
60          77              22
82         107              21
91         118              19
13          16              19
76          99              17
12          15              17
